In [1]:
import pandas as pd
import numpy as np

In [2]:
sales = pd.read_csv("../../../Data/sales.csv")
products = pd.read_csv("../../../Data/products.csv")
stores = pd.read_csv("../../../Data/stores.csv")
category = pd.read_csv("../../../Data/category.csv")
warranty = pd.read_csv("../../../Data/warranty.csv")

In [4]:
master_sales = sales.merge(products, on="product_id")
master_sales = master_sales.merge(category, on="category_id")
master_sales = master_sales.merge(stores, on="store_id")
master_sales = master_sales.merge(warranty, on="sale_id")

In [5]:
master_sales.columns

Index(['sale_id', 'sale_date', 'store_id', 'product_id', 'quantity',
       'product_name', 'category_id', 'launch_date', 'price', 'category_name',
       'store_name', 'city', 'country', 'claim_id', 'claim_date',
       'claim_status'],
      dtype='object')

Identify the Top 10 Stores based on Total Revenue

In [9]:
top_5_products = (
    master_sales
    .groupby("product_name")
    .agg(
        total_quantity=("quantity", "sum")
    )
    .sort_values(by="total_quantity", ascending=False)
    .head(5)
)

print(top_5_products)

                     total_quantity
product_name                       
AirTag                         5097
iPhone SE (3rd Gen)            1899
Apple One                      1380
iPad Air (4th Gen)             1374
iPhone 14                      1321


Identify the Top 10 Stores based on Total Revenue

In [12]:
master_sales["revenue"] = master_sales["quantity"] * master_sales["price"]

In [13]:
top_10_stores = (
    master_sales
    .groupby("store_name")
    .agg(
        total_revenue=("revenue", "sum")
    )
    .sort_values(by="total_revenue", ascending=False)
    .head(10)
)

print(top_10_stores)

                            total_revenue
store_name                               
Apple Barcelona                   4824981
Apple Mall of the Emirates        4545114
Apple Dubai Mall                  4427897
Apple Ankara                      1863064
Apple Istanbul                    1793999
Apple Milan                       1361696
Apple New Delhi                   1322485
Apple Regent Street                294096
Apple Munich                       258549
Apple Lyon                         178296


Create a Country-wise Business Performance Report containing Revenue, Quantity Sold, Average
Revenue per Sale, and Warranty Claim Rate.

In [17]:
country_report = (
    master_sales
    .groupby("country")
    .agg(
        total_revenue=("revenue", "sum"),
        quantity_sold=("quantity", "sum"),
        average_revenue_per_sale=("revenue", "mean"),
        total_sales=("sale_id", "count"),
        warranty_claims=("claim_status", lambda x: (x == "Free Replaced").sum())
    )
)

country_report["warranty_claim_rate"] = (
    country_report["warranty_claims"]
    / country_report["total_sales"]
) * 100

country_report = country_report[
    [
        "total_revenue",
        "quantity_sold",
        "average_revenue_per_sale",
        "warranty_claim_rate"
    ]
]

print(country_report)

             total_revenue  quantity_sold  average_revenue_per_sale  \
country                                                               
France              321368            512                959.307463   
Germany             434956            744                678.558502   
India              1322485           2255                590.131638   
Italy              1361696           1804                805.260792   
Netherlands         143587            243                880.901840   
Spain              4824981           6640                797.253966   
Turkey             3657063           6157                593.968329   
UAE                8973011          12924                759.394973   
UK                  537030           1740                308.637931   

             warranty_claim_rate  
country                           
France                  0.000000  
Germany                 0.000000  
India                  96.251673  
Italy                  18.450621  
Netherla

Perform a complete Category Performance Analysis including Revenue, Quantity, Product Count, and
Warranty Claims, then rank all categories from best to worst.

In [22]:
category_report = (
    master_sales
    .groupby("category_name")
    .agg(
        total_revenue=("revenue", "sum"),
        quantity_sold=("quantity", "sum"),
        product_count=("product_name", "nunique"),
        warranty_claims=("claim_status", lambda x: (x == "Approved").sum())
    )
    .sort_values(by="total_revenue", ascending=False)
)

category_report["rank"] = category_report["total_revenue"].rank(
    ascending=False,
    method="dense"
)



In [24]:
category_report = category_report[
    [
        "rank",
        "total_revenue",
        "quantity_sold",
        "product_count",
        "warranty_claims"
    ]
]

print(category_report)

                      rank  total_revenue  quantity_sold  product_count  \
category_name                                                             
Smartphone             1.0       12740404          14916             21   
Laptop                 2.0        3638906           1694              4   
Tablet                 3.0        3075935           5705              8   
Wearable               4.0         934458           2342              3   
Desktop                5.0         454719            381              2   
Audio                  6.0         266027           1063              3   
Subscription Service   7.0         251340           1416              2   
Accessory              8.0         147813           5097              1   
Streaming Device       9.0          59249            331              1   
Smart Speaker         10.0           7326             74              1   

                      warranty_claims  
category_name                          
Smartphone         

Prepare the Final Business Dataset for Power BI by creating a clean, analysis-ready DataFrame
containing:

Sale ID

Sale Date

Product Name

Category

Store Name

Country

Quantity

Price

Revenue

Warranty Status

Month

Quarter

Year

Store Performance

Price Category

This dataset should require no additional cleaning before importing into Power BI.

In [30]:
master_sales["revenue"] = master_sales["quantity"] * master_sales["price"]

In [31]:
master_sales["sale_date"] = pd.to_datetime(master_sales["sale_date"])

master_sales["month"] = master_sales["sale_date"].dt.month_name()
master_sales["quarter"] = master_sales["sale_date"].dt.quarter
master_sales["year"] = master_sales["sale_date"].dt.year

In [32]:
def price_category(price):
    if price >= 2000:
        return "Premium"
    elif price >= 1000:
        return "Mid Range"
    else:
        return "Budget"

master_sales["price_category"] = master_sales["price"].apply(price_category)

In [33]:
store_revenue = (
    master_sales.groupby("store_name")["revenue"]
    .transform("sum")
)

total_revenue = master_sales["revenue"].sum()

master_sales["store_contribution"] = (
    store_revenue / total_revenue
) * 100


def store_performance(x):
    if x >= 20:
        return "Excellent"
    elif x >= 10:
        return "Good"
    else:
        return "Poor"

master_sales["store_performance"] = (
    master_sales["store_contribution"]
    .apply(store_performance)
)

In [34]:
powerbi_dataset = master_sales[
    [
        "sale_id",
        "sale_date",
        "product_name",
        "category_name",
        "store_name",
        "country",
        "quantity",
        "price",
        "revenue",
        "claim_status",
        "month",
        "quarter",
        "year",
        "store_performance",
        "price_category"
    ]
]

print(powerbi_dataset.head())

     sale_id  sale_date          product_name category_name      store_name  \
0  OID-59217 2023-12-31  Apple Watch Series 5      Wearable  Apple Istanbul   
1  OID-59218 2023-12-31  Apple Watch Series 5      Wearable  Apple Istanbul   
2  OID-59219 2023-12-31             iPhone 11    Smartphone  Apple Istanbul   
3  OID-59220 2023-12-31         iPhone 11 Pro    Smartphone  Apple Istanbul   
4  OID-59221 2023-12-31         iPhone 11 Pro    Smartphone  Apple Istanbul   

  country  quantity  price  revenue   claim_status     month  quarter  year  \
0  Turkey         1    399      399  Free Replaced  December        4  2023   
1  Turkey         1    399      399  Free Replaced  December        4  2023   
2  Turkey         1    699      699  Free Replaced  December        4  2023   
3  Turkey         1    999      999  Free Replaced  December        4  2023   
4  Turkey         1    999      999  Free Replaced  December        4  2023   

  store_performance price_category  
0            

In [35]:
powerbi_dataset.to_csv("powerbi_dataset.csv", index=False)

1. Final dataset is completely cleaned and Power BI ready.

2. Revenue is pre-calculated, reducing report calculations.

3. Month, Quarter, and Year enable easy time-based analysis.

4. Store Performance allows quick comparison of high and low performing stores.

5. Price Category helps analyze customer buying behavior across different price segments.

6. Dataset contains all key business dimensions:
   Product, Store, Country, Revenue, Warranty, and Time.

7. No additional preprocessing is required before importing into Power BI.